# DMEPOS by Referring Provider & Services data loading and cleaning

This notebook combines and cleans the Medicare Durable Medical Equipment, Devices & Supplies (DMEPOS) - by Referring Provider and Service datasets from the Centers of Medicare & Medicaid Services (CMS) for program years 2021 through 2023. Data cleaning and preprocessing steps include the imputation of missing values, normalization of text fields, and appending program year to each record. Finally, data field headings are transformed into snake case for consistency with other datasets.

For information regarding each variable, please refer to the data dictionary downloadable from [DMEPOS - by Referring Provider and Service](https://data.cms.gov/provider-summary-by-type-of-service/medicare-durable-medical-equipment-devices-supplies/medicare-durable-medical-equipment-devices-supplies-by-referring-provider-and-service).

Cells converted to RAW NB Convert are not necessary to run to understand work performed nor run this notebook without exception.

## Import raw files

In [1]:
import pandas as pd

files_names = ['/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy21_rfrhpr.csv',
               '/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy22_rfrhpr.csv',
               '/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy23_rfrhpr.csv']

rfrhpr = {}

dy = 20
for file_name in files_names:
    dy += 1

    # Create DataFrame from the file content
    df = pd.read_csv(file_name,dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
    rfrhpr[dy] = df

Verify the number of records in each file.

In [2]:
len21 = len(rfrhpr[21])
len22 = len(rfrhpr[22])
len23 = len(rfrhpr[23])
print(f'{len21}, {len22}, {len23}')

1516153, 1457378, 1439587


### Change these to code cells and run for additional validation.

## Combine and Clean Datasets

In [3]:
# combine dfs
df_raw = pd.concat([rfrhpr[21],rfrhpr[22],rfrhpr[23]],axis=0)
df_raw.head()

,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplrs,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Portable gaseous oxygen system, rental; includ...",Y,5,NaN,16,16,46.336250,20.097500,14.857500,15.280000
1,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Oxygen concentrator, single delivery port, cap...",Y,6,NaN,19,19,360.770000,98.223158,72.843684,79.753158
2,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Standard hemi (low seat) wheelchair,Y,1,NaN,11,11,92.000000,39.230000,31.385455,33.552727
3,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,"Elevating leg rests, pair (for use with capped...",Y,1,NaN,11,11,20.000000,10.210909,8.169091,8.456364
4,1003000480,Rothchild,Kevin,B,MD,I,12605 E 16th Ave,NaN,Aurora,CO,...,"Oxygen concentrator, single delivery port, cap...",Y,4,NaN,11,13,272.003846,80.513846,64.407692,84.701538


In [4]:
# check columns which have missing data
df_nulltots = df_raw.isnull().sum()
df_nulltots
# blank first names can be assumed to be entity type code = ‘O’

Rfrg_NPI                          0
Rfrg_Prvdr_Last_Name_Org          0
Rfrg_Prvdr_First_Name            42
Rfrg_Prvdr_MI               1267759
Rfrg_Prvdr_Crdntls           179868
Rfrg_Prvdr_Ent_Cd                 0
Rfrg_Prvdr_St1                    0
Rfrg_Prvdr_St2              3166492
Rfrg_Prvdr_City                   0
Rfrg_Prvdr_State_Abrvtn           0
Rfrg_Prvdr_State_FIPS             0
Rfrg_Prvdr_Zip5                   0
Rfrg_Prvdr_RUCA_Cat            3509
Rfrg_Prvdr_RUCA                3509
Rfrg_Prvdr_RUCA_Desc           3509
Rfrg_Prvdr_Cntry                  0
Rfrg_Prvdr_Spclty_Cd          36480
Rfrg_Prvdr_Spclty_Desc            0
Rfrg_Prvdr_Spclty_Srce            0
RBCS_Lvl                          0
RBCS_Id                           0
RBCS_Desc                         0
HCPCS_CD                          0
HCPCS_Desc                        0
Suplr_Rentl_Ind                   0
Tot_Suplrs                        0
Tot_Suplr_Benes             3077843
Tot_Suplr_Clms              

### Convert the following cell to code and run to see how we determined imputing null credentials was inappropriate.

### Investigation of null specialty codes
The field `Rfrg_Prvdr_Spclty_Cd` indicating the numeric referring provider specialty code from CMS's Medicare Provider and Supplier Taxonomy Crosswalk has null values; whereas, the field `Rfrg_Prvdr_Spclty_Desc` indicating a text description of the referring provider specialty does not. We investigate the descriptions associated with null specialty codes to determine a rational method of imputation.

In [5]:
# descriptions of missing specialty codes
pd.Series(df_raw[df_raw.Rfrg_Prvdr_Spclty_Cd.isnull()]['Rfrg_Prvdr_Spclty_Desc'].unique()).sort_values()

8                                        Anesthesiology
16                                        Clinic/Center
43                           Clinical Neuropsychologist
10                               Colon & Rectal Surgery
12                                            Counselor
30                                              Dentist
23                                   Emergency Medicine
2                                       Family Medicine
27                          General Acute Care Hospital
5                                      General Practice
24                      Health Maintenance Organization
44                                          Home Health
46                                 Integrative Medicine
0                                     Internal Medicine
6                                        Legal Medicine
22                             Licensed Practical Nurse
42                         Local Education Agency (LEA)
37                                              

#### The following cell was converted to markdown for documentation purposes.
Initial investigation missed the specialty descriptions 'Internal Medicine,' 'General Practice,' 'Anesthesiology', 'Emergency Medicine', 'Otolaryngology', 'Dentist', and 'Obstetrics & Gynecology' which are associated with missing specialty codes.

#### Convert the following cell to code and run to see how we determined that speciatly "Psychologist, Clinical" maps to two codes.

### Perform text normalization on provider credentials

In [6]:
import re
# df_raw['Rfrg_Prvdr_Crdntls'].sort_values()
# replace NaN credentials with empty string
df_raw['Rfrg_Prvdr_Crdntls'] = df_raw['Rfrg_Prvdr_Crdntls'].fillna('')
# normalize credentials to lowercase and remove punctuation
df_raw['Rfrg_Prvdr_Crdntls'] = df_raw['Rfrg_Prvdr_Crdntls'].apply(lambda x: re.sub(r'[^a-zA-Z]','',x).lower())
display(df_raw['Rfrg_Prvdr_Crdntls'].unique())
print(df_raw['Rfrg_Prvdr_Crdntls'].nunique())

array(['md', 'do', 'arnp', ..., 'dpmpsc', 'rnlpcfnp', 'msnanpaprnbc'],
      dtype=object)

3486


### Perform data cleaning on geographic attributes

#### Use the 2010 Rural-Urban Commuting Area Codes and ZIP codes table from the US Department of Agriculture to build a mapping of zip codes to rural-urban commuting area (RUCA) codes for imputing null RUCA values.

In [7]:
# build mapping of zip codes to RUCA
# use 2010 data to keep consistent with methodology per data dictionary
import csv

zip_map = {}

with open('/dsa/groups/casestudycf25/team02/bronze/RUCA2010zipcode.csv', mode='r', newline='') as file:
    csv_reader = csv.DictReader(file)
    # print(csv_reader.fieldnames)
    # row = next(csv_reader)
    # print(re.sub(r"'","",row[csv_reader.fieldnames[0]]))
    for row in csv_reader:
        # write RUCA2 code (value) into zip_map with ZIP_CODE as key
        zip_map[re.sub(r"'","",row[csv_reader.fieldnames[0]])] = float(row['RUCA2']) # Access data using column names


In [8]:
# build mapping of RUCA to urban-rural designation and description per https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
code_map = {1.:['Urban','Metropolitan area core: primary flow within an urbanized area of 50,000 and greater'],
            1.1:['Urban','Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater'],
            2.:['Urban','Metropolitan area high commuting: primary flow 30% or more to a urbanized area of 50,000 and greater'],
            2.1:['Urban','Secondary flow 30% to <50% to a larger urbanized area of 50,000 and greater'],
            3.:['Urban','Metropolitan area low commuting: primary flow 10% to <30% to a urbanized area of 50,000 and greater'],
            4.:['Urban','Micropolitan area core: primary flow within an urban cluster of 10,000 to 49,999'],
            4.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            5.:['Urban','Micropolitan high commuting: primary flow 30% or more to a urban cluster of 10,000 to 49,999'],
            5.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            6.:['Urban','Micropolitan low commuting: primary flow 10% to <30% to a urban cluster of 10,000 to 49,999'],
            7.:['Urban','Small town core: primary flow within an urban cluster of 2,500 to 9,999'],
            7.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            7.2:['Urban','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            8.:['Urban','Small town high commuting: primary flow 30% or more to a urban cluster of 2,500 to 9,999'],
            8.1:['Urban','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            8.2:['Urban','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            9.:['Urban','Small town low commuting: primary flow 10% to <30% to a urban cluster of 2,500 to 9,999'],
            10.:['Rural','Rural areas: primary flow to a tract outside a urbanized area of 50,000 and greater or UC'],
            10.1:['Rural','Secondary flow 30% to <50% to a urbanized area of 50,000 and greater'],
            10.2:['Rural','Secondary flow 30% to <50% to a urban cluster of 10,000 to 49,999'],
            10.3:['Rural','Secondary flow 30% to <50% to a urban cluster of 2,500 to 9,999'],
            99.:['Unknown','Unknown']}
            

In [9]:
missing_zips = df_raw[df_raw.Rfrg_Prvdr_RUCA_Cat.isnull()]['Rfrg_Prvdr_Zip5'].unique()
missing_zips

array(['85222', '21264', '85273', '96278', '20307', '99588', '38163',
       '35870', '83886', '96306', '01016', '46258', '09617', '96271',
       '09574', '21423', '23291', '20462', '11111', '23710', '75073',
       '99574', '85239', '75284', '09094', '09180', '22334', '96273',
       '09464', '80262', '96264', '03681', '10064', '09012', '85218',
       '11535', '50200', '20596', '44193', '85242', '86306', '02192',
       '97313', '97881', '31939', '58303', '75258', '96617', '40852',
       '46092', '96362', '23521', '39126', '17712', '37992', '09112',
       '18154', '31514', '00115', '08228', '02031', '09175', '94410',
       '32886', '10571', '60059', '00000', '54502', '48277', '61837',
       '85228', '09824', '99686', '34595', '27759', '71605', '32323',
       '96328', '09104', '29277', '31610', '99286', '99243', '48351',
       '10087', '98341', '85232', '72836', '96368', '96310', '96350',
       '86027', '96224', '02154', '56703', '63195', '76560', '02078',
       '39432', '008

In [10]:
# some zip codes are missing from RUCA2010zipcode.csv
for missing_zip in missing_zips:
    try:
        print(f"{missing_zip} maps to {zip_map[missing_zip]}")
    except:
        zip_map[missing_zip] = 99. # map zip code to 99 for unknown

21264 maps to 1.0
99588 maps to 10.0
38163 maps to 1.0
23291 maps to 1.0
99574 maps to 10.0
75284 maps to 1.0
22334 maps to 1.0
80262 maps to 1.0
44193 maps to 1.0
32886 maps to 1.0
48277 maps to 1.0
99686 maps to 10.0
32323 maps to 2.0
10087 maps to 1.0
63195 maps to 1.0
02241 maps to 1.0
06030 maps to 1.0
06927 maps to 1.0
06102 maps to 1.0
06050 maps to 1.0
06409 maps to 1.0
06504 maps to 1.0
06520 maps to 1.0
06856 maps to 1.0
06134 maps to 1.0
06250 maps to 4.1
06039 maps to 10.0
06904 maps to 1.0
46277 maps to 1.0
06487 maps to 1.0
06782 maps to 1.0
06777 maps to 10.0
06230 maps to 2.0
06890 maps to 1.0
06824 maps to 1.0
06825 maps to 1.0
35246 maps to 1.0
44194 maps to 1.0
48267 maps to 1.0


In [11]:
# link missing RUCA by zip code from table at https://www.ers.usda.gov/data-products/rural-urban-commuting-area-codes
        
df_clean = df_raw
df_clean.Rfrg_Prvdr_RUCA = df_clean.Rfrg_Prvdr_RUCA.fillna(df_clean.Rfrg_Prvdr_Zip5.map(zip_map))

In [12]:
# map in missing RUCA_Cat and Desc
target_columns = ['Rfrg_Prvdr_RUCA_Cat', 'Rfrg_Prvdr_RUCA_Desc']

for i, col in enumerate(target_columns):
    df_clean[col] = df_clean.apply(lambda row: code_map[row['Rfrg_Prvdr_RUCA']][i] if pd.isna(row[col]) else row[col], axis=1)

### Impute missing specialty codes
The mapping of specialty descriptions to their codes from the Medicare Provider and Supplier Taxonomy Crosswalk published by CMS was created manually based on personal judgement. Any description that lacked an obvious counterpart in the crosswalk or that was so ambiguous as to potentially match several specialty codes was left blank.

In [13]:
# manually map specialty codes to Rfrg_Prvdr_Spclty_Desc using Medicare Provider and Supplier Taxonomy Crosswalk dataset from
# https://data.cms.gov/provider-characteristics/medicare-provider-supplier-enrollment/medicare-provider-and-supplier-taxonomy-crosswalk
specialty_map = {'Anesthesiology': '5','Clinic/Center': '70','Clinical Neuropsychologist': '86','Colon & Rectal Surgery': '28','Counselor': 'E2','Dentist': 'C5','Emergency Medicine': '93','Family Medicine': '8',
                 'General Acute Care Hospital': 'A0','General Practice': '1','Health Maintenance Organization': '58','Home Health': 'A4','Integrative Medicine': '',
                 'Internal Medicine': '11','Legal Medicine': '','Licensed Practical Nurse': '89','Local Education Agency (LEA)': '','Midwife': '42',
                 'Military Health Care Provider': 'A0','Naturopath': '','Neurological Surgery': '14','Neuromusculoskeletal Medicine, Sports Medicine': '23','Obstetrics & Gynecology': '16',
                 'Oral & Maxillofacial Surgery': '19','Orthopaedic Surgery': '20','Otolaryngology': '4','Pain Medicine': '72','Pediatrics': '37','Pedorthist': 'B2',
                 'Personal Emergency Response Attendant': '','Pharmacist': 'A5','Pharmacy Technician': 'A5','Physical Medicine & Rehabilitation': '25',
                 'Plastic Surgery': '24','Prosthetist': '56','Psychiatry & Neurology': '26','Psychoanalyst': '62','Psychologist': '62',
                 'Registered Nurse': '89','Respite Care': '','Sleep Specialist, PhD': 'C0','Social Worker': '80','Specialist': '',
                 'Specialist/Technologist, Other': '75','Student in an Organized Health Care Education/Training Program': '',
                 'Surgery': '2','Thoracic Surgery (Cardiothoracic Vascular Surgery)': '33'}

In [14]:
df_clean.Rfrg_Prvdr_Spclty_Cd = df_clean.Rfrg_Prvdr_Spclty_Cd.fillna(df_clean.Rfrg_Prvdr_Spclty_Desc.map(specialty_map))

### Change specialty descriptions to snake case

In [15]:
# prepare the to_snake_case function
def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

In [16]:
# transform the data
df_clean['Rfrg_Prvdr_Spclty_Desc'] = df_clean.Rfrg_Prvdr_Spclty_Desc.apply(to_snake_case)

### Impute nulls in the Total Supplier Beneficiaries field

In [17]:
# impute missing values in a rational manner
# Real totals can be determined from data described on this webpage:
# https://www.cms.gov/data-research/cms-data/limited-data-set-lds-files/physician/supplier-procedure-summary-psps
# As it costs $400 to obtain we opt for the imputation methodology of Khoshgoftaar & Johnson i.e. imputing 5 for missing Tot_Benes and Tot_Suplr_Benes

df_clean['Tot_Suplr_Benes'] = df_clean['Tot_Suplr_Benes'].fillna(5)

In [18]:
df_clean.isnull().sum() # make sure remaining nulls are sensible

Rfrg_NPI                          0
Rfrg_Prvdr_Last_Name_Org          0
Rfrg_Prvdr_First_Name            42
Rfrg_Prvdr_MI               1267759
Rfrg_Prvdr_Crdntls                0
Rfrg_Prvdr_Ent_Cd                 0
Rfrg_Prvdr_St1                    0
Rfrg_Prvdr_St2              3166492
Rfrg_Prvdr_City                   0
Rfrg_Prvdr_State_Abrvtn           0
Rfrg_Prvdr_State_FIPS             0
Rfrg_Prvdr_Zip5                   0
Rfrg_Prvdr_RUCA_Cat               0
Rfrg_Prvdr_RUCA                   0
Rfrg_Prvdr_RUCA_Desc              0
Rfrg_Prvdr_Cntry                  0
Rfrg_Prvdr_Spclty_Cd              0
Rfrg_Prvdr_Spclty_Desc            0
Rfrg_Prvdr_Spclty_Srce            0
RBCS_Lvl                          0
RBCS_Id                           0
RBCS_Desc                         0
HCPCS_CD                          0
HCPCS_Desc                        0
Suplr_Rentl_Ind                   0
Tot_Suplrs                        0
Tot_Suplr_Benes                   0
Tot_Suplr_Clms              

### Append years

In [19]:
import numpy as np
# arrays of year values
yrs_21 = np.full(len21, 2021)
yrs_22 = np.full(len22, 2022)
yrs_23 = np.full(len23, 2023)

# concatenate
yrs = np.concatenate((yrs_21, yrs_22, yrs_23))

# add year column to the df
df_clean['Year'] = yrs

df_clean.head()

,Rfrg_NPI,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,...,Suplr_Rentl_Ind,Tot_Suplrs,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt,Year
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,5,5.0,16,16,46.336250,20.097500,14.857500,15.280000,2021
1,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,6,5.0,19,19,360.770000,98.223158,72.843684,79.753158,2021
2,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,1,5.0,11,11,92.000000,39.230000,31.385455,33.552727,2021
3,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Y,1,5.0,11,11,20.000000,10.210909,8.169091,8.456364,2021
4,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,Y,4,5.0,11,13,272.003846,80.513846,64.407692,84.701538,2021


### Add AAPC service descriptions

In [20]:
import re
# NOTE that pharmacy codes for Pharmacy Supply and Dispensing Fees and Miscellaneous Drug and New Technology Codes overlap
# we do not anticipate signifcant issues

def get_hcpcs_range(string):
    # create a dictionary of regexes that map to hcpcs ranges
    hcpcs_dic = {'Skin Substitute Device':[r'A4001'],
                 'Injection and Infusion Supplies':[r'A42[012].',r'A423[012]'],
                 'Replacement Batteries':[r'A423[^012]'],
                 'Other Supplies Including Diabetes Supplies and Contraceptives':[r'A42[456789].'],
                 'Access Catheters and Drug Delivery Systems':[r'A430.'],
                 'Incontinence Devices and Supplies':[r'A43[12345].',r'A4360',r'A5[12]..'],
                 'Ostomy Pouches and Supplies':[r'A436[^0]',r'A43[789].',r'A44[0123].',r'A50..'],
                 'Various Medical Supplies Including Tapes and Surgical Dressings':[r'A44[56789].',r'A45..',r'A460.'],
                 'Respiratory Supplies and Equipment':[r'A46[12].'],
                 'Replacement Parts':[r'A463.',r'A4640'],
                 'Diagnostic Radiopharmaceuticals':[r'A464[12]'],
                 'Other Supplies':[r'A464[^012]',r'A465[012].'],
                 'Dialysis Equipment and Supplies':[r'A465[^012]',r'A46[6789].',r'A47..',r'A48..',r'A49..'],
                 'Diabetic Footwear':[r'A55..'],
                 'Miscellaneous Dressing and Wound Supplies':[r'A6[01]..',r'A620[^9]'],
                 'Foam Dressings':[r'A6209',r'A621[012345]'],
                 'Gauze Dressings':[r'A621[6789]',r'A622.',r'A623[0123]'],
                 'Hydrocolloid Dressings':[r'A623[^0123]',r'A624[01]'],
                 'Hydrogel Dressings':[r'A624[^01]'],
                 'Other Dressings, Coverings, and Wound Treatment Supplies':[r'A62[56789].',r'A63..',r'A640.',r'A641[012]'],
                 'Bandages':[r'A641[^012]',r'A64[23456].'],
                 'Compression Garments and Stockings':[r'A65..',r'A66..'],
                 'Breathing Aids':[r'A70..'],
                 'Tracheostoma Supplies':[r'A75..'],
                 'Miscellaneous Supplies and Equipment':[r'A9[123]..'],
                 'Diagnostic and Therapeutic Radiopharmaceuticals':[r'A9[5678]..'],
                 'Miscellaneous DME Supplies and Services':[r'A9...'],
                 'Enteral Feeding Supplies and Equipment':[r'B40..'],
                 'Enteral Formulas and Additives':[r'B41[012345].',r'B416[012]'],
                 'Parenteral Solutions and Supplies':[r'B416[^0123]',r'B41[789].',r'B4[^01]..',r'B5[01]..',r'B5200'],
                 'Nutrition Infusion Pumps and Supplies Not Otherwise Classified, NOC':[r'B9...'],
                 'Walking Aids and Attachments':[r'E01[012345].'],
                 'Sitz Bath/Equipment':[r'E016[012]'],
                 'Commode Chair and Supplies':[r'E016[^012]',r'E017.'],
                 'Pressure Mattresses, Pads, and Other Supplies':[r'E01[89].'],
                 'Heat, Cold, and Light Therapies':[r'E02[0123].'],
                 'Bathing Supplies':[r'E024.'],
                 'Hospital Beds and Associated Supplies':[r'E02[56789].',r'E03..'],
                 'Oxygen Delivery Systems and Related Supplies':[r'E04..'],
                 'Intermittent Positive Pressure Breathing Devices':[r'E0500'],
                 'Electronic Positional Obstructive Sleep Apnea Treatment':[r'E0530'],
                 'Humidifiers and Nebulizers with Related Equipment':[r'E05[56789].',r'E060[01]'],
                 'Breast Pumps':[r'E060[234]'],
                 'Other Breathing Aids':[r'E060[56]'],
                 'Monitoring Equipment':[r'E060[789]',r'E061.',r'E0620'],
                 'Patient Lifts and Support Systems':[r'E062[^0]',r'E063.',r'E064[012]'],
                 'Pneumatic Compressors and Appliances':[r'E06[56].',r'E067[^89]'],
                 'Non Pneumatic Compressors and Appliances':[r'E067[89]',r'E068[0123]'],
                 'ULTRAVIOLET LIGHT THERAPY SYSTEMS':[r'E069.'],
                 'Safety Devices':[r'E070.',r'E071[01]'],
                 'Intravaginal Device and Supplies for Pelvic Floor Muscles Strengthening':[r'E071[56]'],
                 'Stimulation Devices':[r'E07[23456].',r'E0770'],
                 'Infusion Pumps and Supplies':[r'E077[6789]',r'E07[89].'],
                 'Traction and Other Orthopedic Devices':[r'E08..',r'E09[01234].'],
                 'Wheelchair Accessories':[r'E09[56789].',r'E10[012].',r'E103[^789]'],
                 'Transport Chairs':[r'E103[789]'],
                 'Fully Reclining Wheelchairs':[r'E10[567].'],
                 'Hemi-Wheelchairs':[r'E108[3456]'],
                 'Lightweight, High-strength Wheelchairs':[r'E108[789]',r'E1090'],
                 'Heavy Duty, Wide Wheelchairs':[r'E109[23]'],
                 'Semi-reclining Wheelchairs':[r'E11[01].'],
                 'Standard Wheelchairs':[r'E11[3456].'],
                 'Amputee Wheelchairs':[r'E11[789].',r'E1200'],
                 'Other Wheelchairs and Accessories':[r'E122[^9]'],
                 'Pediatric Wheelchairs':[r'E1229',r'E123.'],
                 'Lightweight Wheelchairs':[r'E12[4567].'],
                 'Heavy Duty and Special Wheelchairs':[r'E12[89].'],
                 'Whirlpool Baths':[r'E13[01].'],
                 'Accessories for Oxygen Delivery Devices':[r'E13[56789].',r'E14..'],
                 'Dialysis Systems and Accessories':[r'E15..',r'E16..'],
                 'Jaw Motion Rehabilitation Systems':[r'E17..'],
                 'Extension/Flexion Rehabilitation Devices':[r'E18..'],
                 'Communication Boards':[r'E1902'],
                 'Miscellaneous Pumps and Monitors':[r'E20..',r'E21[01].',r'E2120'],
                 'Virtual Reality Cognitive Behavioral Therapy Device (CBT)':[r'E1905'],
                 'Manual Wheelchair Accessories':[r'E22[^9].',r'E229[12345]'],
                 'Power Wheelchair Accessories':[r'E229[89]',r'E23..'],
                 'Wound Therapy Pumps':[r'E2402'],
                 'Speech Generating Devices, Software, and Accessories':[r'E25..'],
                 'Wheelchair Seat and Back Cushions':[r'E26[01].',r'E262[^6789]'],
                 'Wheelchair Mobile Arm Supports':[r'E262[6789]',r'E263.'],
                 'Vaccine Administration':[r'G000.',r'G0010'],
                 'Psychotherapy, Pre-Exposure Prophylaxis Counseling and Community Health Integration Services':[r'G001[^0]',r'G002[01234]'],
                 'Analysis of Semen Specimen':[r'G0027'],
                 'MIPS Measures':[r'G002[89]',r'G00[345].',r'G006[^89]'],
                 'Professional Services for Drug Infusion':[r'G006[89]',r'G0070'],
                 'Telemed Services':[r'G0071'],
                 'Home Care Management Services':[r'G007[6789]',r'G008[^89]'],
                 'Initial Visit for Professional Services':[r'G008[89]',r'G0090'],
                 'Screening Examinations and Disease Management Training':[r'G01[01].',r'G012[01234]'],
                 'Miscellaneous Diagnostic and Therapeutic Services':[r'G012[789]',r'G02..',r'G03[^789].',r'G037[012]'],
                 'Hospital Observation and Emergency Services':[r'G037[89]',r'G038.'],
                 'Other Emergency Services':[r'G0390'],
                 'Alcohol and Substance Abuse Assessments':[r'G039[67]'],
                 'Sleep Studies, In Home':[r'G039[89]',r'G0400'],
                 'Initial Services for Medicare Enrollment':[r'G040[2345]'],
                 'Followup Telehealth Consultations':[r'G040[678]'],
                 'Psychological Services':[r'G0409',r'G041[01]'],
                 'Fracture Treatment':[r'G041[2345]'],
                 'Gross and Microscopic Examinations, Prostate Biopsy':[r'G0416'],
                 'Face-to-Face Educational Services':[r'G042[01]'],
                 'Cardiac and Pulmonary Rehabilitation Services':[r'G042[23]'],
                 'Initial Telehealth Consultations':[r'G042[567]'],
                 'Filler Procedures':[r'G042[89]'],
                 'Laboratory Screening Tests':[r'G043[2345]'],
                 'Counseling, Screening, and Prevention Services':[r'G043[89]',r'G044.',r'G045[01]'],
                 'Miscellaneous Services':[r'G045[^01]',r'G046[^6789]'],
                 'Federally Qualified Health Center (FQHC) Visits':[r'G046[6789]',r'G0470'],
                 'Other Services':[r'G047[^0]',r'G04[89].',r'G05..',r'G06..'],
                 'Quality Measures for Cataract Surgery':[r'G09..'],
                 'Drugs, Administered by Injection':[r'J[^789]...',r'J70..',r'J71[^789].',r'J717[^6789]'],
                 'Clotting Factors':[r'J71[789].',r'J721.'],
                 'Contraceptive Systems':[r'J729.',r'J730[^89]'],
                 'Miscellaneous Drugs':[r'J730[89]',r'J73[^0].',r'J74..',r'Q40[78].'],
                 'Immunosuppressive Drugs':[r'J75..'],
                 'Inhalation Solutions':[r'J76[^9].'],
                 'Drugs, Not Otherwise Classified':[r'J7699',r'J7[789]..',r'J8[01234]..'],
                 'Chemotherapy Drugs, Oral Administration':[r'J8[^01234]..'],
                 'Chemotherapy Drugs':[r'J9...'],
                 'Wheelchairs, Components, and Accessories':[r'K0[01]..'],
                 'INFUSION PUMPS AND SUPPLIES':[r'K0[45]..',r'K060[^6789]'],
                 'Automated External Defibrillator and Supplies':[r'K060[6789]'],
                 'Miscellaneous DME and Accessories':[r'K06[6789].',r'K07..'],
                 'Power Operated Vehicles':[r'K080.',r'K081[012]'],
                 'Wheelchairs, Power Operated':[r'K081[^012]',r'K08[^01].'],
                 'Customized DME, Other Than Wheelchair':[r'K0900'],
                 'Components, Accessories and Supplies':[r'K1...'],
                 'Cervical Orthotics':[r'L01[^89].'],
                 'Cervical Orthotics Multi-post Collar':[r'L01[89].',r'L0200'],
                 'Thoracic Rib Belts':[r'L0220'],
                 'Thoracic-lumbar-sacral (TLSO) Orthotics':[r'L04..'],
                 'Sacral Orthotics':[r'L062[1234]'],
                 'Lumbar Orthotics':[r'L062[567]'],
                 'Lumbar-sacral Orthotics':[r'L062[89]',r'L063.',r'L0640'],
                 'Lumbar Orthotics Sagittal Control':[r'L064[12]'],
                 'Lumbar-sacral Orthotics Sagittal Control':[r'L064[^012]',r'L065.'],
                 'Cervical-thoracic-lumbar-sacral Orthotics':[r'L07..'],
                 'Cervical Halo Procedures':[r'L08..'],
                 'Accessories for Spinal Orthotics':[r'L09..'],
                 'Scoliosis Orthotic Devices':[r'L1[01]..'],
                 'Low-profile Additions, Thoracic-lumbar-sacral Orthotics':[r'L12..'],
                 'Other Scoliosis and Spinal Orthotics and Procedures':[r'L1[34]..'],
                 'Hip Orthotics':[r'L16..'],
                 'Legg Perthes Orthotics':[r'L17..'],
                 'Knee Orthotics':[r'L18..'],
                 'Ankle-foot Orthotics':[r'L19..',r'L21[01].'],
                 'Knee-ankle-foot Orthotics':[r'L20[0123].'],
                 'Hip-knee-ankle-foot Orthotics':[r'L20[^0123].'],
                 'Knee-ankle-foot Orthotics':[r'L21[23].'],
                 'Additions, Lower Extremity, Fracture Orthotics':[r'L21[89]..'],
                 'Additions, Lower Extremity Orthotics':[r'L2[23]..'],
                 'Orthotic Additions to Knee Joints':[r'L24..'],
                 'Additions, Weight-bearing, Lower Extremities':[r'L25[^6789].',r'L26..'],
                 'Other Lower Extremity Additions':[r'L2[789]..'],
                 'Foot Inserts, Removable':[r'L30[0123].'],
                 'Foot Arch Supports':[r'L30[^0123].'],
                 'Repositioning Foot Orthotics':[r'L31..'],
                 'Orthopedic Shoes':[r'L320[^89]'],
                 'Surgical Boots':[r'L320[89]',r'L321[01]'],
                 'Benesch Boots':[r'L321[234]'],
                 'Other Orthopedic Footwear':[r'L321[56789]',r'L32[23456].'],
                 'Shoe Lifts':[r'L33[0123].'],
                 'Shoe Wedges':[r'L33[^0123].',r'L34[012].'],
                 'Shoe Heels':[r'L34[^0129].'],
                 'Other Orthopedic Shoe Additions':[r'L35..'],
                 'Orthosis Transfers':[r'L36[01234].'],
                 'Shoulder Orthotics':[r'L36[567].'],
                 'Elbow Orthotics':[r'L37[^6789].',r'L376[012]'],
                 'Elbow-wrist-hand Orthotics':[r'L376[3456]'],
                 'Wrist-hand-finger Orthotics':[r'L38..',r'L390[01234]'],
                 'Wrist-hand Orthotics':[r'L390[5678]'],
                 'Additional Miscellaneous Orthotics, Upper Extremities':[r'L39[12345].'],
                 'Shoulder-elbow-wrist-hand Orthotics':[r'L396.',r'L397[0123]'],
                 'Shoulder-elbow-wrist-hand-finger Orthotics':[r'L397[5678]'],
                 'Fracture, Addition, and Unspecified Orthotics, Upper Extremities':[r'L39[89].'],
                 'Orthotic Replacement Parts or Repair':[r'L4[012]..'],
                 'Other Lower Extremity Orthotics':[r'L4[3456]..'],
                 'Partial Foot Prosthetics':[r'L50[012].'],
                 'Ankle Prosthetics':[r'L50[56].'],
                 'Below the Knee Prosthetics':[r'L510[^6789]'],
                 'Knee Disarticulation Prosthetics':[r'L51[56].'],
                 'Above the Knee Prosthetics':[r'L52[0123].'],
                 'Hip Disarticulation Prosthetics':[r'L52[567].'],
                 'Endoskeletal Prosthetics, Lower Limbs':[r'L52[89].',r'L53..'],
                 'Prosthetic Fitting, Immediate Postsurgical or Early, Lower Limbs':[r'L54..'],
                 'Supply, Initial Prosthesis':[r'L550.'],
                 'Supply, Preparatory Prosthesis':[r'L55[^0].',r'L560.'],
                 'Endoskeletal Prosthetic Additions, Lower Extremities':[r'L561[^89]'],
                 'Test Socket Prosthetic Additions, Lower Extremities':[r'L561[89]',r'L562[^9]'],
                 'Various Prosthetic Sockets':[r'L5629',r'L56[34].',r'L565[0123]'],
                 'Socket Insert, Suspensions, and Other Prosthetic Additions':[r'L565[^0123]',r'L56..'],
                 'Replacement Sockets':[r'L570[0123]'],
                 'Custom-shaped Protective Covers':[r'L570[4567]'],
                 'Exoskeletal Knee-shin System Additions':[r'L57[^89].',r'L5780'],
                 'Vacuum Pumps, Lower Limb Prosthetic Additions':[r'L578[123]'],
                 'Other Exoskeletal Additions':[r'L578[^0123]',r'L579.'],
                 'Endoskeletal Knee or Hip System Additions':[r'L58..',r'L59[^6789].',r'L596[^789]'],
                 'Ankle and/or Foot Prosthetics and Additions':[r'L596[89]',r'L59[789].'],
                 'Partial Hand Prosthetics':[r'L60[0123].'],
                 'Wrist Disarticulation, Hand Prosthetics':[r'L605.'],
                 'Below Elbow, Forearm and Hand Prosthetics':[r'L61..'],
                 'Elbow Disarticulation, Forearm and Hand Prosthetics':[r'L620.'],
                 'Above Elbow, Forearm and Hand Prosthetics':[r'L6250'],
                 'Shoulder Disarticulation, Arm and Hand Prosthetics':[r'L63[012].'],
                 'Interscapular Thoracic, Arm, and Hand Prosthetics':[r'L63[567].'],
                 'Prosthetic Fitting, Immediate Postsurgical or Early, Upper Limbs':[r'L638.'],
                 'Molded Socket Endoskeletal Prosthetic System, Upper Limbs':[r'L64..',r'L65[^89].'],
                 'Preparatory Prosthetic, Upper Limbs':[r'L65[89].'],
                 'Upper Extremity Prosthetic Additions':[r'L66..',r'L6700'],
                 'Terminal Devices and Additions':[r'L670[^012]',r'L68[^89].',r'L688[012]'],
                 'Replacement Sockets, Upper Limbs':[r'L688[345]'],
                 'Hand Restoration Prosthetics and Additions':[r'L689.',r'L69[01].'],
                 'External Power Upper Limb Prosthetics':[r'L69[^0189]'],
                 'Electric Hand or Hook and Additions':[r'L70..'],
                 'Electronic Elbow and Additions':[r'L7[12]..'],
                 'Batteries and Accessories':[r'L736.'],
                 'Additions for Upper Extremity Prosthetics':[r'L740[^789]'],
                 'Upper Extremity Prosthetics, Not Otherwise Specified (NOS)':[r'L7499'],
                 'Prosthetic Repair':[r'L75[12].'],
                 'Prosthetic Donning Sleeve':[r'L7600'],
                 'Gasket or Seal with Prosthetic':[r'L7700'],
                 'Penile Prosthetics':[r'L79..'],
                 'Breast Prosthetics and Accessories':[r'L80[0123].'],
                 'Facial and External Ear Prosthetics':[r'L804.'],
                 'Hernia Trusses':[r'L83..'],
                 'Prosthetic Sheaths, Socks, and Shrinkers':[r'L84[^9].'],
                 'Unlisted Prosthetic Procedures':[r'L8499'],
                 'Voice Prosthetics and Accessories':[r'L85..'],
                 'Prosthetic Breast Implant':[r'L8600'],
                 'Bulking Agents':[r'L860[34567]'],
                 'Implantable Eye and Ear Prosthetics and Accessories':[r'L860[89]',r'L86[12].'],
                 'Implantable Hand and Feet Prosthetics':[r'L86[345].'],
                 'Vascular Implants':[r'L8670'],
                 'Implantable Neurostimulators and Components':[r'L867[89]',r'L868.'],
                 'Miscellaneous Orthotic and Prosthetic Services and Supplies':[r'L869.',r'L8[789]..',r'L9[^9]..',r'L9900'],
                 'Miscellaneous Drugs and Tests':[r'Q00..',r'Q01[01234].'],
                 'Chemotherapy Anti-emetic Medications':[r'Q01[5678].'],
                 'COVID-19 Infusion Therapy':[r'Q02..'],
                 'Ventricular Assist Devices':[r'Q04..',r'Q050.'],
                 'Pharmacy Supply and Dispensing Fees':[r'Q051.',r'Q052[01]'],
                 'Miscellaneous Drug and New Technology Codes':[r'Q051[^01234].',r'Q0[6789]..',r'Q1...',r'Q201.',r'Q202[^9]'],
                 'Influenza Virus Vaccines':[r'Q203.'],
                 'Other Drugs and Service Fees':[r'Q20[^0123].',r'Q2[^0]..',r'Q3...'],
                 'Cast and Splint Supplies':[r'Q40[^6789]..'],
                 'Skin Substitutes and Biologicals':[r'Q4[1234]..'],
                 'Spectacle Frames':[r'V20..'],
                 'Lenses, Single Vision':[r'V21..'],
                 'Lenses, Bifocals':[r'V22..'],
                 'Lenses, Trifocal':[r'V23..'],
                 'Lenses, Aspherical and Variable Sphericity':[r'V24..'],
                 'Assorted Contact Lenses':[r'V25..'],
                 'Low and Near Vision Aids':[r'V26[01].'],
                 'Eye Prosthetics and Services':[r'V262.'],
                 'Lenses, Intraocular':[r'V263.'],
                 'Vision Services':[r'V27..']}
    for key, value in hcpcs_dic.items():
        for pattern in value:
            if re.fullmatch(pattern, string):
                aapc = key
                return aapc
            else:
                continue
    return 'unknown'

In [21]:
df_clean['aapc_desc'] = df_clean['HCPCS_CD'].apply(get_hcpcs_range)

Make sure we caught all HCPCS.

In [22]:
df_clean[df_clean['aapc_desc']=='unknown']['HCPCS_CD'].unique()

array([], dtype=object)

How many AAPC descriptions are there?

In [23]:
descriptions = df_clean['aapc_desc'].unique()
len(descriptions)

142

### Impute Total Medicare payment amounts by provider, year, service, and rental indicator

In [24]:
df_clean['tot_suplr_mdcr_pymt_amt'] = df_clean['Tot_Suplr_Srvcs'] * df_clean['Avg_Suplr_Mdcr_Pymt_Amt']

## Calculate z-score of total amount received for each service each year

In [25]:
# calculate means and std devs by year and aapc_desc grouping
df_clean['avg_suplr_mdcr_pymt_amt_hcpcs'] = df_clean.groupby(['Year','HCPCS_CD','Suplr_Rentl_Ind'])['tot_suplr_mdcr_pymt_amt'].transform('mean')
df_clean['std_suplr_mdcr_pymt_amt_hcpcs'] = df_clean.groupby(['Year','HCPCS_CD','Suplr_Rentl_Ind'])['tot_suplr_mdcr_pymt_amt'].transform('std')

In [26]:
# z-score of tot_suplr_mdcr_pymt_amt by year and aapc_desc grouping 
df_clean['tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z-scr'] = (df_clean['tot_suplr_mdcr_pymt_amt'] - df_clean['avg_suplr_mdcr_pymt_amt_hcpcs'])/df_clean['std_suplr_mdcr_pymt_amt_hcpcs']

In [27]:
# drop the mean and std dev columns
df_clean = df_clean.drop(columns=['avg_suplr_mdcr_pymt_amt_hcpcs','std_suplr_mdcr_pymt_amt_hcpcs'])

In [28]:
# if there was insufficient data to compute std_dev then the z-score will be na
# set these to zero
df_clean['tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z-scr'] = df_clean['tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z-scr'].fillna(0)

## Calculate Ratios of providers' average allowed to the national average for same HCPCS and rental indicator

In [29]:
df_clean['Avg_HCPCS_Rentl_Ind_Avg_Suplr_Mdcr_Alowd_Amt'] = df_clean.groupby(['Year','HCPCS_CD','Suplr_Rentl_Ind'])['Avg_Suplr_Mdcr_Alowd_Amt'].transform('mean')

In [30]:
df_clean['Avg_Suplr_Mdcr_Alowd_Amt_HCPCS_Rentl_Ind_Rat'] = df_clean['Avg_Suplr_Mdcr_Alowd_Amt']/df_clean['Avg_HCPCS_Rentl_Ind_Avg_Suplr_Mdcr_Alowd_Amt']

In [31]:
# drop the mean column
df_clean = df_clean.drop(columns='Avg_HCPCS_Rentl_Ind_Avg_Suplr_Mdcr_Alowd_Amt')

## Write-out and validate combined datasets

In [32]:
# convert column headings to snake case
df_clean = df_clean.rename(columns={col: to_snake_case(col) for col in df_clean.columns})

In [33]:
# write to csv and store in silver folder
df_clean.to_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrhpr.csv', index=False) 

In [34]:
# read clean csv back in for validation
# import pandas as pd

df = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrhpr.csv',dtype={'rfrg_prvdr_state_fips':str,'rfrg_prvdr_zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
df.head()

,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,rfrg_prvdr_st2,rfrg_prvdr_city,rfrg_prvdr_state_abrvtn,...,tot_suplr_srvcs,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,avg_suplr_mdcr_pymt_amt,avg_suplr_mdcr_stdzd_amt,year,aapc_desc,tot_suplr_mdcr_pymt_amt,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr,avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,16,46.336250,20.097500,14.857500,15.280000,2021,Oxygen Delivery Systems and Related Supplies,237.72,-0.425926,0.955638
1,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,19,360.770000,98.223158,72.843684,79.753158,2021,Accessories for Oxygen Delivery Devices,1384.03,-0.347468,0.942689
2,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,11,92.000000,39.230000,31.385455,33.552727,2021,"Wheelchairs, Components, and Accessories",345.24,-0.390508,0.985031
3,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,11,20.000000,10.210909,8.169091,8.456364,2021,"Wheelchairs, Components, and Accessories",89.86,-0.409305,1.022396
4,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,13,272.003846,80.513846,64.407692,84.701538,2021,Accessories for Oxygen Delivery Devices,837.30,-0.419031,0.772725


Verify the number of records was preserved.

In [35]:
check_len = len21 + len22 + len23
df_len = len(df)
print(f'{check_len} versus {df_len}')

4413118 versus 4413118


### Convert the following cell to code and run to see the data types of the full dataset.